## Install and load required packages

In [2]:
# Install if needed
#install.packages("logistf")
#install.packages("dplyr")

library(logistf)
library(dplyr)

also installing the dependencies 'ucminf', 'ordinal', 'pan', 'jomo', 'glmnet', 'mitml', 'mice'





The downloaded binary packages are in
	/var/folders/0x/crbr62xs7z56zg0dzh5548w00000gn/T//Rtmpqw7rZd/downloaded_packages



Attaching package: 'dplyr'


The following objects are masked from 'package:stats':

    filter, lag


The following objects are masked from 'package:base':

    intersect, setdiff, setequal, union




## Import data and run Firth logistic regression

In [ ]:
# Import data
# Define path to data
path_to_folder_with_data <- "/Users/nicholasbartelo/Documents/Zhao-Lab-AMR-Paper-main/Statistical Testing/Input File"

# Read in data
annotated_mz_data <- read.csv(
  file.path(path_to_folder_with_data, "annotated_mz_only_post_processed_maplet_data.csv"),
  row.names = 1
)

# Ensure Group is a factor
annotated_mz_data$Group <- as.factor(annotated_mz_data$Group)

# Initialize storage
significant_results_list <- list()
all_results_list <- list()

# Continuous variables
continuous_vars <- names(annotated_mz_data)[!(names(annotated_mz_data) %in% c("Group", "Organism", "date_measured", "batch_measured"))]

warning_count <- 0

# Loop over organisms
for (organism in unique(annotated_mz_data$Organism)) {
  
  subset_data <- annotated_mz_data[annotated_mz_data$Organism == organism, ]
  pvalues <- numeric(length(continuous_vars))
  warning_occurred <- FALSE
  
  for (i in seq_along(continuous_vars)) {
    metabolite <- continuous_vars[i]
    
    model <- suppressWarnings(
      logistf(as.formula(paste("Group ~", metabolite)), 
              data = subset_data,
              plcontrol = logistf.control(maxit = 100))
    )
    
    if (length(warnings()) > 0) {
      warning_occurred <- TRUE
    }
    
    pvalues[i] <- summary(model)$prob[2]
  }
  
  if (warning_occurred) {
    warning_count <- warning_count + 1
  }
  
  results <- data.frame(
    organism = organism,
    metabolite = continuous_vars,
    p_value = pvalues
  )
  
  results$fdr_adjusted_p_value <- p.adjust(results$p_value, method = "fdr")

  significant_results <- results %>%
    filter(fdr_adjusted_p_value < 0.05) %>%
    select(organism, metabolite, fdr_adjusted_p_value)
  
  all_results_list[[organism]] <- results
  significant_results_list[[organism]] <- significant_results
}

all_significant_results <- bind_rows(significant_results_list)
all_results <- bind_rows(all_results_list)

logistf(formula = as.formula(paste("Group ~", metabolite)), data = subset_data, 
    plcontrol = logistf.control(maxit = 100))

Model fitted by Penalized ML
Coefficients:
                   coef  se(coef) lower 0.95 upper 0.95     Chisq         p
(Intercept)  0.06904095 0.2143774 -0.3534452   0.490760 0.1035901 0.7475632
X341.3045359 0.30707834 0.2482366 -0.1729095   0.811416 1.5638463 0.2111029
             method
(Intercept)       2
X341.3045359      2

Method: 1-Wald, 2-Profile penalized log-likelihood, 3-None

Likelihood ratio test=1.563846 on 1 df, p=0.2111029, n=89
Wald test = 1.800476 on 1 df, p = 0.1796549logistf(formula = as.formula(paste("Group ~", metabolite)), data = subset_data, 
    plcontrol = logistf.control(maxit = 100))

Model fitted by Penalized ML
Coefficients:
                     coef  se(coef) lower 0.95 upper 0.95        Chisq
(Intercept)  0.0008184909 0.2477924 -0.4964677  0.4857694 1.089189e-05
X688.1566289 0.2059552854 0.2451225 -0.2702395  0.7108702 7.176799

## Save results

In [8]:
# Clean metabolite names
all_results$metabolite <- substr(all_results$metabolite, 2, nchar(all_results$metabolite))

# Sort
all_results_sorted <- all_results %>%
  arrange(organism, fdr_adjusted_p_value)

# Save full results
#write.csv(all_results_sorted, "firth_logistic_regression_results_all_organisms.csv", row.names = FALSE)

# Remove specific organisms
all_results_final <- all_results %>%
  filter(organism != "Escherichia coli" & organism != "Klebsiella pneumoniae")

# Sort again
all_results_final_sorted <- all_results_final %>%
  arrange(organism, fdr_adjusted_p_value)

# Save filtered results
#write.csv(all_results_final_sorted, "firth_logistic_regression_results.csv", row.names = FALSE)